#Overall architecture

from IPython.display import display, HTML


<pre style="
  background:#1e1e2e;
  color:#cdd6f4;
  padding:24px;
  border-radius:12px;
  font-size:14px;
  line-height:2;
  font-family:monospace;
">
👤 Widget: User Input (Market / Time Range)
              │
              ▼
🎯 Python: Orchestrator — prepare data context
              │
              ▼
📊 Agent: Overview — generate headline metrics
              │
              ▼
🔍 Agent: Anomaly Detector — check for irregularity
         │                    │
   Has anomaly           No anomaly
         │                    │
         ▼                    │
🔬 Agent: Deep Dive           │
   break down drivers         │
         │                    │
         └──────────┬─────────┘
                    ▼
📝 Agent: Summary — executive summary + recommendations
</pre>

In [92]:
import pandas as pd
import numpy as np
import random
from datetime import date, timedelta
import json
from dateutil.relativedelta import relativedelta
import os
from google.colab import drive
import google.generativeai as genai
from google.colab import userdata

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/Gen-AI/sales-agent/data'
os.makedirs(save_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [93]:
# API setup
api_key = userdata.get('API_Agent')
genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash-lite")

#Load Data and Schema

In [94]:
df_revenue = pd.read_csv(f"{save_dir}/monthly_revenue.csv")
df_customer = pd.read_csv(f"{save_dir}/customers.csv")
df_churn = pd.read_csv(f"{save_dir}/monthly_churn.csv")

with open(f"{save_dir}/schema.json", "r") as file:
    schema = file.read()

# Custom queries (Data analysts write own queries)

In [95]:
start_date = '2022-01-01'
end_date = '2024-12-31'

##Query library and function

In [96]:
import duckdb

# Register dataframes as SQL tables
con = duckdb.connect()
con.register('customers', df_customer)
con.register('monthly_revenue', df_revenue)
con.register('monthly_churn', df_churn)

def run_query(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

In [97]:
df_revenue.head()

,customer_id,month,market,segment,plan,product,mrr,event_type
0,SLEEK-0001,2023-04,HK,Startup,Starter,Incorporation,790.65,new
1,SLEEK-0001,2023-04,HK,Startup,Starter,Accounting,42.64,new
2,SLEEK-0001,2023-04,HK,Startup,Starter,Compliance,28.66,new
3,SLEEK-0001,2023-05,HK,Startup,Starter,Accounting,40.95,active
4,SLEEK-0001,2023-05,HK,Startup,Starter,Compliance,28.02,active


In [98]:
df_customer.head()

,customer_id,signup_date,market,segment,plan,acquisition_channel,is_foreign_founder,headcount,cac,churned,churn_date,ltv
0,SLEEK-0001,2023-04-03,HK,Startup,Starter,Organic,False,6,75.69,False,NaN,2258.43
1,SLEEK-0002,2022-01-18,SG,Startup,Starter,Organic,False,3,84.80,False,NaN,3356.15
2,SLEEK-0003,2023-02-11,SG,Startup,Starter,Partner,False,2,132.92,True,2024-01,1537.63
3,SLEEK-0004,2023-12-15,SG,Startup,Starter,Partner,False,6,178.39,False,NaN,1652.99
4,SLEEK-0005,2022-01-02,SG,SME,Growth,Partner,False,22,166.38,True,2023-03,11979.12


#Orchastrator

In [99]:
def orchastrator(market= ['All'], start_date=start_date, end_date=end_date) -> dict:

  start_dt = pd.to_datetime(start_date)
  extended_start = (start_dt - pd.DateOffset(months=1)).strftime('%Y-%m-%d')

  #market filter
  if market == ['All']:

    market_list = tuple(df_revenue['market'].unique())

  else:
    market_list = tuple(market)

  mrr_trend=f"""

  select
    month
    ,market
    ,segment
    ,plan
    ,product
    ,event_type
    ,sum(mrr) as monthly_revenue
    ,count(distinct customer_id) as customers
  from monthly_revenue
  where
    product != 'Incorporation'
    and month between substring('{extended_start}',1,7) and substring('{end_date}',1,7)
    and market in {market_list}
  group by month, market, segment, plan, product, event_type

  """

  churn_query = f"""

  select
    churn_month as month
    ,market
    ,segment
    ,plan
    ,churn_reason_category
    ,sum(mrr_lost) as monthly_mrr_lost
    ,count(customer_id) as customers_lost
  from monthly_churn
  where
    churn_month between substring('{start_date}',1,7) and substring('{end_date}',1,7)
    and market in {market_list}
  group by churn_month, market, segment, plan, churn_reason_category

  """

  cac_ltv_query = f"""
  SELECT
      acquisition_channel,
      market,
      segment,
      AVG(cac) as avg_cac,
      AVG(ltv) as avg_ltv,
      AVG(ltv / cac) as ltv_cac_ratio,
      COUNT(customer_id) as customers
  FROM customers
  WHERE market IN {market_list}
  AND signup_date BETWEEN '{start_date}' AND '{end_date}'
  GROUP BY acquisition_channel, market, segment
  ORDER BY avg_cac DESC
  """

  df_mrr_trend = run_query(mrr_trend)
  df_churn_trend = run_query(churn_query)
  df_cac_trend = run_query(cac_ltv_query)

  return {
      'params': {'market': market_list, 'start_date': start_date, 'end_date': end_date},
      'schema': schema,
      'mrr_data': df_mrr_trend.to_dict(orient='records'),
      'churn_data': df_churn_trend.to_dict(orient='records'),
      'cac_data': df_cac_trend.to_dict(orient='records')
  }

In [100]:
result = orchastrator(market= ['All'], start_date=start_date, end_date=end_date)

In [101]:
test_df = pd.DataFrame(result['mrr_data'])
test_df.head()

,month,market,segment,plan,product,event_type,monthly_revenue,customers
0,2023-05,HK,Startup,Starter,Compliance,active,1257.95,45
1,2023-06,HK,Startup,Starter,Compliance,active,1288.10,46
2,2023-08,HK,Startup,Starter,Accounting,active,2145.21,51
3,2023-09,HK,Startup,Starter,Accounting,active,2307.83,55
4,2023-09,HK,Startup,Starter,Compliance,active,1537.20,55


#Overview agent

In [102]:
def agent_overview(context: dict) -> dict:
    schema = json.dumps(json.loads(context['schema'])['tables'])

    # 1. Prepare MRR data - Filter out the 'extended' previous month for aggregate stats
    mrr_full = pd.DataFrame(context['mrr_data'])
    start_month = context['params']['start_date'][:7]
    # Filter mrr_data to only include the user's selected period for total sums
    mrr_filtered = mrr_full[mrr_full['month'] >= start_month]

    total_mrr = mrr_filtered['monthly_revenue'].sum()

    # Calculate performance by dimensions (Market, Segment, etc.)
    top_mrr_market = mrr_filtered.groupby('market')['monthly_revenue'].sum().idxmax()
    top_mrr_segment = mrr_filtered.groupby('segment')['monthly_revenue'].sum().idxmax()
    top_mrr_plan = mrr_filtered.groupby('plan')['monthly_revenue'].sum().idxmax()
    top_mrr_product = mrr_filtered.groupby('product')['monthly_revenue'].sum().idxmax()

    # Share percentages
    mkt_pct = (mrr_filtered.groupby('market')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
    segment_pct = (mrr_filtered.groupby('segment')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
    plan_pct = (mrr_filtered.groupby('plan')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
    product_pct = (mrr_filtered.groupby('product')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()

    # Growth calculation using the full range (previous month vs last month)
    mrr_by_month = mrr_full.groupby('month')['monthly_revenue'].sum().sort_index()
    first_mrr = mrr_by_month.iloc[0] # This is the BOP (previous month)
    last_mrr = mrr_by_month.iloc[-1]
    mrr_growth = ((last_mrr - first_mrr) / first_mrr * 100).round(1)

    # 2. Prepare Churn data
    churn_data = pd.DataFrame(context['churn_data'])
    total_churn_customer = churn_data['customers_lost'].sum()

    # Get active customers for the LATEST month only
    latest_month = mrr_full['month'].max()
    active_customer_latest = int(mrr_full[mrr_full['month'] == latest_month]['customers'].sum())

    # Churn breakdown
    top_churn_reasons = churn_data.groupby('churn_reason_category')['customers_lost'].sum().idxmax()
    top_churn_market = churn_data.groupby('market')['customers_lost'].sum().idxmax()

    # 3. CAC / LTV Data
    cac_data = pd.DataFrame(context['cac_data'])
    avg_cac = cac_data['avg_cac'].mean().round(2)
    avg_ltv = cac_data['avg_ltv'].mean().round(2)
    avg_ltv_cac = cac_data['ltv_cac_ratio'].mean().round(2)
    top_cac_channel = cac_data.groupby('acquisition_channel')['avg_cac'].mean().idxmax()

    # 4. Construct Prompt
    # Note: We now include specific "Churn Rate" context
    prompt = f"""
    You are a data analyst at an accounting SaaS company.
    Provide a high-level overview of the business performance.

    ### Parameters
    Markets: {context['params']['market']} | Period: {context['params']['start_date']} to {context['params']['end_date']}

    ### Pre-calculated Metrics (Use these exactly)
    - Total MRR in Period: ${total_mrr:,.0f}
    - MRR Growth (vs Start of Period): {mrr_growth}%
    - Top Market/Segment: {top_mrr_market} / {top_mrr_segment}
    - Total Customers Lost: {total_churn_customer}
    - Current Active Customers: {active_customer_latest}
    - Market Share: {mkt_pct}
    - Segment Share: {segment_pct}
    - Product Share: {product_pct}
    - Top Churn Reason: {top_churn_reasons}
    - Average CAC: ${avg_cac:,.0f}
    - Average LTV: ${avg_ltv:,.0f}
    - LTV:CAC Ratio: {avg_ltv_cac}x
    - Most Expensive Channel: {top_cac_channel}

    ### Detailed Data
    MRR Trend:
    {mrr_filtered.to_string(index=False)}

    Churn Data:
    {churn_data.to_string(index=False)}

    ## Instructions
    - Reference specific numbers in EVERY sentence.
    - Format: $1,234 for revenue, 12.3% for percentages.
    - Focus on the "Churn Rate" context: With {active_customer_latest} current actives and {total_churn_customer} lost, explain the retention health.

    ────────────────────────────────────────────────────
    Overall Performance:
    [2 sentences on MRR trend and growth]

    ────────────────────────────────────────────────────
    Key Highlights:
    [3 sentences on trends across market, segment, or plan]

    ────────────────────────────────────────────────────
    Revenue Mix:
    [2 sentences on revenue drivers]

    ────────────────────────────────────────────────────
    Customer Churn & Retention:
    [2 sentences on churn reasons and market impact]

    ────────────────────────────────────────────────────
    Acquisition Efficiency:
    [2 sentences on CAC vs LTV and channel performance]

    ────────────────────────────────────────────────────
    Watch Out:
    [1 sentence on the most concerning data point]
    """

    return model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.3)
    ).text

#Anomaly Detector Agent

In [103]:
def agent_anomaly_detector(context: dict,
                            churn_threshold=50,
                            mrr_neg_months=2,
                            ltv_cac_threshold=3,
                            revenue_mix_threshold=0.7) -> dict:

    # 1. Prepare base dataframes
    mrr_data = pd.DataFrame(context['mrr_data'])
    churn_data = pd.DataFrame(context['churn_data'])

    # Create a monthly customer snapshot (used as the denominator for churn rates)
    # This includes the extended month fetched by the Orchestrator
    mrr_monthly_snapshot = mrr_data.groupby('month')['customers'].sum()

    # 2. Calculate MRR monthly growth rate
    mrr_by_month = (
        mrr_data.groupby('month')['monthly_revenue'].sum()
        .reset_index()
        .sort_values('month')
    )
    mrr_by_month['mrr_growth'] = mrr_by_month['monthly_revenue'].pct_change() * 100

    # 3. Calculate precise Churn Rate % (Numerator: current churn / Denominator: previous month active)
    churn_by_month = (
        churn_data.groupby('month')['customers_lost'].sum()
        .reset_index().rename(columns={'customers_lost': "total_churn"})
        .sort_values('month')
    )

    def calc_rate(row):
        # Identify the previous month string (YYYY-MM)
        prev_m = (pd.to_datetime(row['month']) - relativedelta(months=1)).strftime('%Y-%m')
        bop_count = mrr_monthly_snapshot.get(prev_m, 0)
        return (row['total_churn'] / bop_count * 100) if bop_count > 0 else 0

    churn_by_month['churn_rate'] = churn_by_month.apply(calc_rate, axis=1)

    # Calculate Churn Rate baseline (3-month rolling average of the Rate)
    churn_by_month['3m_rate_avg'] = churn_by_month['churn_rate'].rolling(3, min_periods=1).mean().shift(1)
    # Calculate variance percentage (Current rate vs. Baseline average)
    churn_by_month['churn_vs_baseline'] = (
        ((churn_by_month['churn_rate'] - churn_by_month['3m_rate_avg']) / churn_by_month['3m_rate_avg'] * 100)
        .round(1).fillna(0)
    )

    # 4. Prepare churn breakdown by market (assists AI in identifying Drill Down markets)
    churn_by_market = (
        churn_data.groupby(['month', 'market'])['customers_lost'].sum()
        .reset_index()
    )

    # 5. Calculate Revenue Mix for concentration risk detection
    total_mrr = mrr_data['monthly_revenue'].sum()
    market_pct = (mrr_data.groupby('market')['monthly_revenue'].sum() / total_mrr).to_dict()
    segment_pct = (mrr_data.groupby('segment')['monthly_revenue'].sum() / total_mrr).to_dict()
    plan_pct = (mrr_data.groupby('plan')['monthly_revenue'].sum() / total_mrr).to_dict()

    # 6. Construct AI Prompt with translated instructions
    prompt = f"""
    You are a senior SaaS data analyst. Analyze the following metrics and identify anomalies based on the rules.

    Market: {context['params']['market']} | Period: {context['params']['start_date']} to {context['params']['end_date']}

    ### 1. MRR Trend (All Markets)
    {mrr_by_month.to_string(index=False)}

    ### 2. Churn Rate Trend (Based on Beginning-of-Period customers)
    {churn_by_month[['month', 'total_churn', 'churn_rate', 'churn_vs_baseline']].to_string(index=False)}

    ### 3. Churn by Market (Raw numbers for drill down)
    {churn_by_market.to_string(index=False)}

    ### 4. Revenue Mix (Concentration Risk)
    - Market Share: {market_pct}
    - Segment Share: {segment_pct}
    - Plan Share: {plan_pct}

    ### Anomaly Detection Rules:
    - MRR: Flag if MRR growth is negative for {mrr_neg_months} consecutive months.
    - Churn: Flag if 'churn_vs_baseline' is > +{churn_threshold}%. (Indicates the churn RATE has significantly spiked).
    - LTV:CAC: Flag if current LTV/CAC ratio is < {ltv_cac_threshold}x.
    - Revenue Mix: Flag if concentration is > {revenue_mix_threshold * 100}%.

    Return ONLY a JSON object:
    {{
      "has_anomaly": true,
      "anomalies": [
        {{
          "metric": "MRR | Churn | LTV:CAC | Revenue Mix",
          "month": "YYYY-MM or null",
          "observation": "Describe using exact numbers (e.g., Churn rate spiked to 4.5%, which is 60% above baseline)",
          "severity": "High | Medium",
          "possible_cause": "Hypothesis",
          "drill_down_market": "Primary market responsible or null"
        }}
      ],
      "summary": "One-sentence executive summary"
    }}
    """

    # 7. Call LLM and parse JSON output
    raw = model.generate_content(prompt, generation_config=genai.GenerationConfig(temperature=0.0)).text
    clean = raw.replace("```json", "").replace("```", "").strip()
    result = json.loads(clean)

    # 8. Python-side guardrails (Verify AI claims against actual data)
    churn_lookup = churn_by_month.set_index('month')['churn_vs_baseline'].to_dict()
    neg_growth = mrr_by_month['mrr_growth'] < 0
    consecutive_neg = neg_growth & neg_growth.shift(1, fill_value=False)

    filtered = []
    for a in result['anomalies']:
        if a['metric'] == 'Churn':
            if abs(churn_lookup.get(a['month'], 0)) <= churn_threshold:
                continue
        elif a['metric'] == 'MRR':
            if not consecutive_neg.any():
                continue
        filtered.append(a)

    result['anomalies'] = filtered
    result['has_anomaly'] = len(filtered) > 0

    # Print logs to console
    print("=" * 55)
    print("🔍 ANOMALY DETECTOR (Rate-Based)")
    print("=" * 55)
    print(f"\n{result['summary']}\n")

    return result

def agent_anomaly_detector(context: dict,
                            churn_threshold=50,
                            mrr_neg_months=2,
                            ltv_cac_threshold=3,
                            revenue_mix_threshold=0.7,
                            cac_threshold=2) -> dict:

  mrr_data = pd.DataFrame(context['mrr_data'])
  churn_data = pd.DataFrame(context['churn_data'])

  #MRR monthly trend
  mrr_by_month = (
      mrr_data.groupby('month')['monthly_revenue'].sum()
      .reset_index()
      .sort_values('month')
  )

  mrr_by_month['mrr_growth'] = mrr_by_month['monthly_revenue'].pct_change() * 100

  #Churn
  churn_by_month = (
      churn_data.groupby('month')['customers_lost'].sum()
      .reset_index().rename(columns={'customers_lost': "total_churn"})
      .sort_values('month')
  )

  churn_by_month['3m_average'] = churn_by_month['total_churn'].rolling(3, min_periods=1).mean().shift(1)
  churn_by_month['churn_vs_baseline'] = (
      (churn_by_month['total_churn'] - churn_by_month['3m_average'])
      / churn_by_month['3m_average'] * 100
  ).round(1)


  #Churn by market
  churn_by_market = (
      churn_data.groupby(['month', 'market'])['customers_lost'].sum()
      .reset_index()
  )

  # Revenue mix
  total_mrr = mrr_data['monthly_revenue'].sum()
  market_pct = (mrr_data.groupby('market')['monthly_revenue'].sum() / total_mrr).to_dict()
  segment_pct = (mrr_data.groupby('segment')['monthly_revenue'].sum() / total_mrr).to_dict()
  plan_pct = (mrr_data.groupby('plan')['monthly_revenue'].sum() / total_mrr).to_dict()
  product_pct = (mrr_data.groupby('product')['monthly_revenue'].sum() / total_mrr).to_dict()

  prompt=f"""
  You are a data analyst at an accounting SaaS company.

  You are goal is to analyze the data and report any anomalies you observed based on the trends.

  Market: {context['params']['market']} | Period: {context['params']['start_date']} to {context['params']['end_date']}

  #MRR Trend (monthly, all markets)
  {mrr_by_month.to_string(index=False)}

  #Churn Trend(monthly, all markets)
  {churn_by_month.to_string(index=False)}

  #Churn by Market (monthly)
  {churn_by_market.to_string(index=False)}

  #Revenue Mix
  Market share: {market_pct}
  Segment share: {segment_pct}
  Plan share: {plan_pct}
  Product share: {product_pct}

  #Anomaly rules
  - MRR: flag out anomaly if the MRR trend has been negative for {mrr_neg_months} consecutive months
  - Churn: flag out anomaly if churn_vs_baseline is > +{churn_threshold}%. Use exact value from [churn_by_month] table only
  - LTV:CAC < {ltv_cac_threshold}x → flag. LTV:CAC > {ltv_cac_threshold}x is healthy, do NOT flag
  - Revenue Mix: flag ONLY if one market/segment/plan > {revenue_mix_threshold * 100}%. Exclude Payroll from product check
  - CAC: not applicable without customer-level data

  ## Drill Down Rules:
  - For churn spikes: cross-reference churn by market table, set drill_down_market to market with highest customers_lost that month
  - For market concentration: set drill_down_market to that market
  - If not market-specific: set drill_down_market to null

  ## Critical Instructions:
  - Only apply rules listed above, do not invent new rules
  - Do NOT flag if threshold is not strictly exceeded
  - Do NOT include anomaly if your observation says threshold is not met
  - severity must be "High" or "Medium" only

  Return ONLY a JSON object, no explanation, no markdown fences:
  {{
    "has_anomaly": true or false,
    "anomalies": [
      {{
        "metric": "MRR | Churn | LTV:CAC | Revenue Mix",
        "month": "YYYY-MM or null",
        "observation": "one sentence with exact numbers from data",
        "severity": "High | Medium",
        "possible_cause": "one sentence hypothesis",
        "drill_down_market": "SG | HK | AU | UK | null"
      }}
    ],
    "summary": "one sentence summary"
  }}
  """
  raw    = model.generate_content(prompt, generation_config=genai.GenerationConfig(temperature=0.0)).text
  clean  = raw.replace("```json", "").replace("```", "").strip()
  result = json.loads(clean)

  # ── Python filter ─────────────────────────────────────────────────────────
  churn_lookup = churn_by_month.set_index('month')['churn_vs_baseline'].to_dict()
  neg_growth   = mrr_by_month['mrr_growth'] < 0
  consecutive_neg = neg_growth & neg_growth.shift(1, fill_value=False)

  filtered = []
  for a in result['anomalies']:
      if a['metric'] == 'Churn':
          if abs(churn_lookup.get(a['month'], 0)) <= churn_threshold:
              continue
      elif a['metric'] == 'Revenue Mix':
          all_pcts = {**market_pct, **segment_pct, **plan_pct}
          if not any(v > revenue_mix_threshold for v in all_pcts.values()):
              continue
      elif a['metric'] == 'MRR':
          if not consecutive_neg.any():
              continue
      filtered.append(a)

  result['anomalies'] = filtered
  result['has_anomaly'] = len(filtered) > 0

  # ── Print ──────────────────────────────────────────────────────────────────
  print("=" * 55)
  print("🔍 ANOMALY DETECTOR")
  print("=" * 55)
  print(f"\n{result['summary']}\n")

  if result['has_anomaly']:
      print(f"⚠️  {len(result['anomalies'])} anomaly/anomalies detected:\n")
      for a in result['anomalies']:
          month_str  = f"[{a['month']}] " if a['month'] else ""
          market_str = f"  → Drill down: {a['drill_down_market']}" if a.get('drill_down_market') else ""
          print(f"  {a['metric']} {month_str}[{a['severity']}]")
          print(f"  → {a['observation']}")
          print(f"  → Possible cause: {a['possible_cause']}")
          if market_str:
              print(market_str)
          print()
  else:
      print("✅ No anomalies detected — all metrics within normal range")

  print("=" * 55)
  return result

#Deepdive Agent

In [104]:
def agent_deep_dive(anomaly_result: dict, start_date: str, end_date: str) -> dict:

    flagged_markets = set(
        a['drill_down_market']
        for a in anomaly_result['anomalies']
        if a.get('drill_down_market') and a['drill_down_market'] not in [None, 'null', '']
    )

    print('flagged markets: ', flagged_markets)

    if not flagged_markets:
        print("No market-specific anomalies to drill down.")
        return {}

    print('Flagged_markets: ', flagged_markets)

    deep_dive_results = {}

    for market in flagged_markets:

        print('Market: ', market)

        # Re-run orchestrator for specific market
        mkt_ctx = orchastrator(market=[market], start_date=start_date, end_date=end_date)

        # Get anomalies for this market
        mkt_anomalies = [
            a for a in anomaly_result['anomalies']
            if a.get('drill_down_market') == market
        ]

        # Pre-calculate from raw data
        mrr_df   = pd.DataFrame(mkt_ctx['mrr_data'])
        churn_df = pd.DataFrame(mkt_ctx['churn_data'])
        cac_df   = pd.DataFrame(mkt_ctx['cac_data'])

        # MRR breakdown
        total_mrr    = mrr_df['monthly_revenue'].sum()
        segment_pct  = (mrr_df.groupby('segment')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()
        plan_pct     = (mrr_df.groupby('plan')['monthly_revenue'].sum() / total_mrr * 100).round(1).to_dict()

        # Churn breakdown
        churn_by_segment = churn_df.groupby('segment')['customers_lost'].sum().to_dict()
        churn_by_plan    = churn_df.groupby('plan')['customers_lost'].sum().to_dict()
        churn_by_reason  = churn_df.groupby('churn_reason_category')['customers_lost'].sum().to_dict()

        # CAC / LTV
        avg_cac     = cac_df['avg_cac'].mean().round(2)
        avg_ltv_cac = cac_df['ltv_cac_ratio'].mean().round(2)
        top_cac_channel = cac_df.groupby('acquisition_channel')['avg_cac'].mean().idxmax()

        prompt = f"""
        You are a senior sales analyst at Sleek, an accounting SaaS company.
        You are deep diving into the {market} market to identify root causes of detected anomalies.

        ## Detected Anomalies in {market}
        {json.dumps(mkt_anomalies, indent=2)}

        ## {market} Pre-calculated Metrics
        - Total MRR: ${total_mrr:,.0f}
        - MRR by Segment (%): {segment_pct}
        - MRR by Plan (%): {plan_pct}
        - Churn by Segment: {churn_by_segment}
        - Churn by Plan: {churn_by_plan}
        - Churn by Reason: {churn_by_reason}
        - Avg CAC: ${avg_cac:,.0f}
        - LTV:CAC Ratio: {avg_ltv_cac}x
        - Most Expensive Channel: {top_cac_channel}

        ## {market} MRR Trend
        {mrr_df.groupby('month')['monthly_revenue'].sum().reset_index().to_string(index=False)}

        ## {market} Churn Trend
        {churn_df.groupby('month')['customers_lost'].sum().reset_index().to_string(index=False)}

        ## Instructions
        - Focus on identifying WHICH segment / plan / channel is driving the anomaly
        - Cross-reference MRR and churn breakdowns to find the most likely driver
        - Be specific with numbers from the pre-calculated metrics

        Return ONLY a JSON object, no explanation, no markdown fences:
        {{
          "market": "{market}",
          "anomaly_summary": "one sentence describing the anomaly in this market with exact numbers",
          "drivers": [
            {{
              "dimension": "Segment | Plan | Channel | Product",
              "driver": "specific value e.g. Startup, Paid, Starter",
              "observation": "one sentence with exact numbers",
              "contribution": "High | Medium"
            }}
          ],
          "root_cause": "2 sentences synthesizing the most likely root cause with numbers",
          "recommended_action": "one sentence on what to investigate or action to take"
        }}
        """

        raw    = model.generate_content(prompt, generation_config=genai.GenerationConfig(temperature=0.1)).text
        clean  = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(clean)
        deep_dive_results[market] = result

        # Print
        print("=" * 55)
        print(f"🔬 DEEP DIVE — {market}")
        print("=" * 55)
        print(f"\n{result['anomaly_summary']}\n")
        print("── Drivers ──────────────────────────────────────────")
        for d in result['drivers']:
            print(f"  [{d['dimension']}] {d['driver']} ({d['contribution']})")
            print(f"  → {d['observation']}\n")
        print("── Root Cause ───────────────────────────────────────")
        print(f"  {result['root_cause']}\n")
        print("── Recommended Action ───────────────────────────────")
        print(f"  {result['recommended_action']}")
        print("=" * 55)

    return deep_dive_results

#Summary Agent

In [105]:
def agent_summary(context: dict, overview: str, anomaly_result: dict, deep_dive: dict) -> str:

    # Prepare deep dive text
    deep_dive_str = ""
    if deep_dive:
        for market, result in deep_dive.items():
            deep_dive_str += f"\n### {market}\n"
            deep_dive_str += f"Anomaly: {result.get('anomaly_summary', '')}\n"
            deep_dive_str += f"Root Cause: {result.get('root_cause', '')}\n"
            deep_dive_str += f"Recommended Action: {result.get('recommended_action', '')}\n"
    else:
        deep_dive_str = "No market-specific anomalies detected."

    prompt = f"""
    You are a senior sales analyst at an accounting SaaS company.
    Write a concise executive summary for the leadership team.

    ## Parameters
    Market: {context['params']['market']} | Period: {context['params']['start_date']} to {context['params']['end_date']}

    ## Overview Analysis
    {overview}

    ## Anomaly Detection Summary
    {anomaly_result['summary']}

    ## Anomalies Found
    {json.dumps(anomaly_result['anomalies'], indent=2) if anomaly_result['has_anomaly'] else "No anomalies detected."}

    ## Deep Dive Findings
    {deep_dive_str}

    ## Instructions
    - Write for a leadership audience — clear, concise, actionable
    - Must include specific numbers in every section
    - No bullet points, write in prose

    Write exactly these 3 sections separated by a divider:

    ────────────────────────────────────────────────────
    What Happened:
    [2-3 sentences — overall performance + key anomalies found]

    ────────────────────────────────────────────────────
    Why It Happened:
    [2-3 sentences — root causes from deep dive, specific market/segment/plan]

    ────────────────────────────────────────────────────
    What To Do:
    [3 prioritized actions, numbered 1-3, one sentence each]
    """

    result = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.4)
    ).text

    print("=" * 55)
    print("📝 EXECUTIVE SUMMARY")
    print("=" * 55)
    print(result)
    print("=" * 55)

    return result

#Pipeline

In [106]:
def run_pipeline(market=['All'], start_date=start_date, end_date=end_date):

    print("\n" + "=" * 55)
    print("🚀 PIPELINE STARTED")
    print(f"   Market: {market} | {start_date} to {end_date}")
    print("=" * 55 + "\n")

    # Step 1: Orchestrator
    print("⚙️  Step 1: Orchestrating data...\n")
    ctx = orchastrator(market=market, start_date=start_date, end_date=end_date)

    # Step 2: Overview
    print("⚙️  Step 2: Running Overview Agent...\n")
    overview = agent_overview(ctx)

    # Step 3: Anomaly Detector
    print("⚙️  Step 3: Running Anomaly Detector...\n")
    anomaly = agent_anomaly_detector(ctx)

    # Step 4: Deep Dive
    deep_dive = {}
    if anomaly['has_anomaly']:
        print("⚙️  Step 4: Anomaly found — Running Deep Dive Agent...\n")
        deep_dive = agent_deep_dive(anomaly, start_date=start_date, end_date=end_date)
    else:
        print("⚙️  Step 4: No anomalies — Skipping Deep Dive\n")

    # Step 5: Summary
    print("⚙️  Step 5: Running Summary Agent...\n")
    summary = agent_summary(context=ctx, overview=overview, deep_dive=deep_dive, anomaly_result=anomaly)

    # Pre-calculate headline metrics
    mrr_df   = pd.DataFrame(ctx['mrr_data'])
    churn_df = pd.DataFrame(ctx['churn_data'])
    cac_df   = pd.DataFrame(ctx['cac_data'])

    mrr_by_month  = mrr_df.groupby('month')['monthly_revenue'].sum().sort_index()
    curr_mrr      = float(mrr_by_month.iloc[-1])
    prev_mrr      = float(mrr_by_month.iloc[0]) if len(mrr_by_month) > 1 else curr_mrr
    mrr_growth    = round((curr_mrr - prev_mrr) / prev_mrr * 100, 2) if prev_mrr > 0 else 0
    total_churned = int(churn_df['customers_lost'].sum())
    avg_cac       = float(round(cac_df['avg_cac'].mean(), 2))
    avg_ltv       = float(round(cac_df['avg_ltv'].mean(), 2))
    ltv_cac_ratio = float(round(cac_df['ltv_cac_ratio'].mean(), 2))
    last_month    = mrr_df['month'].max()

    #Churn rate: The average churn rate by month
    mrr_monthly = mrr_df.groupby('month')['customers'].sum()
    churn_monthly = churn_df.groupby('month')['customers_lost'].sum()

    selected_months = pd.date_range(start=start_date, end=end_date, freq='MS').strftime('%Y-%m').tolist()

    monthly_rates = []

    #Get monthly numbers for baseline and previous months
    for m in selected_months:
        prev_m = (pd.to_datetime(m) - pd.DateOffset(months=1)).strftime('%Y-%m')

        bop_count = mrr_monthly.get(prev_m, 0)
        lost_count = churn_monthly.get(m, 0)

        if bop_count > 0:
            monthly_rates.append(lost_count / bop_count)

    #Calculate average monthly churn
    if monthly_rates:
        avg_churn_rate = round(np.mean(monthly_rates) * 100, 2)
    else:
        avg_churn_rate = 0.0

    last_month = mrr_df['month'].max()
    active_customers = int(mrr_df[mrr_df['month'] == last_month]['customers'].sum())
    # -------------------------

    print("\n" + "=" * 55)
    print("✅ PIPELINE COMPLETE")
    print("=" * 55 + "\n")

    return {
        'context':   ctx,
        'overview':  overview,
        'anomaly':   anomaly,
        'deep_dive': deep_dive,
        'summary':   summary,
        'headline_metrics': {
            'curr_mrr':          curr_mrr,
            'mrr_growth':        mrr_growth,
            'churned_customers': total_churned,
            'active_customers':  active_customers,
            'churn_rate_pct':    avg_churn_rate,
            'avg_cac':           avg_cac,
            'avg_ltv':           avg_ltv,
            'ltv_cac_ratio':     ltv_cac_ratio,
        }
    }

#Generate demo data

In [107]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/Gen-AI/sales-agent/result'
os.makedirs(save_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [108]:
import json
import time

scenarios = [
    {'market': ['All'], 'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['HK'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['SG'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['AU'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
    {'market': ['UK'],  'start_date': '2022-01-01', 'end_date': '2024-12-31'},
]

for s in scenarios:
    print(f"Running {s['market'][0]}...")
    result = run_pipeline(**s)

    filename = f"{save_dir}/{s['market'][0]}_2022_2024.json"
    with open(filename, 'w') as f:
        json.dump(result, f, indent=2, default=str)

    print(f"Saved {filename}\n")

    time.sleep(30)

Running All...

🚀 PIPELINE STARTED
   Market: ['All'] | 2022-01-01 to 2024-12-31

⚙️  Step 1: Orchestrating data...

⚙️  Step 2: Running Overview Agent...

⚙️  Step 3: Running Anomaly Detector...

🔍 ANOMALY DETECTOR (Rate-Based)

Several significant spikes in churn rate, particularly in April 2023, September-October 2023, and July 2024, indicate potential customer retention issues that require immediate investigation.

⚙️  Step 4: Anomaly found — Running Deep Dive Agent...

flagged markets:  {'SG', 'HK'}
Flagged_markets:  {'SG', 'HK'}
Market:  SG
🔬 DEEP DIVE — SG

SG experienced a significant churn spike in April 2023 (1.06%, 50.2% above baseline) and July 2024 (0.95%, 60.2% above baseline).

── Drivers ──────────────────────────────────────────
  [Segment] Startup (High)
  → Startups account for 20.6% of MRR but 37.5% of churned customers (211 out of 563 total churned customers).

  [Plan] Starter (High)
  → Starter plans represent 4.1% of MRR but 23.6% of churned customers (133 out o

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3034.85ms


🔍 ANOMALY DETECTOR (Rate-Based)

The business exhibits high concentration risk in the UK market and has experienced several significant spikes in churn rate, particularly in 2023 and 2024, warranting further investigation into customer retention strategies.

⚙️  Step 4: Anomaly found — Running Deep Dive Agent...

flagged markets:  {'UK'}
Flagged_markets:  {'UK'}
Market:  UK
🔬 DEEP DIVE — UK

The UK market experienced several high-severity churn anomalies in March 2023 (3.31% churn, 53.7% above baseline), September 2023 (1.00% churn, 70.8% above baseline), October 2023 (1.39% churn, 96.2% above baseline), February 2024 (1.85% churn, 70.3% above baseline), and June 2024 (0.95% churn, 110.5% above baseline), alongside a concerning 100% market share concentration.

── Drivers ──────────────────────────────────────────
  [Segment] Startup (High)
  → Startups represent 13.5% of MRR but account for 21 out of 39 total churned customers, indicating a disproportionately high churn rate within th